# Sync Work-Author Curations

Syncs the work-author half of author curations (claim a work for an author / disclaim a work from an author) from the users Heroku Postgres database to local Databricks Delta tables. Runs in `walden_end2end.yaml` after `Works_Base`; consumed by `ApplyWorkAuthorCurations` after `Author_Matching`.

The names half (set display_name / set full_name) is a sibling pair in `notebooks/authors/`, run from `jobs/authors.yaml`. See `ApplyAuthorNameCurations.ipynb` for the cross-pipeline topology.

**Sources** (typed views over `openalex_users.public.curations`):
- `work_author_claim_curations` — name-anchored claim, `property = 'authorships[raw_author_name="<name>"].author.id'`; the view extracts `raw_author_name TEXT` from the property string
- `work_author_remove_curations` — non-positional sticky disclaim, `property = 'authorships.author.id'`

**Targets**:
- `openalex.works.work_author_claim_curations`
- `openalex.works.work_author_remove_curations`

`WHEN NOT MATCHED BY SOURCE THEN DELETE` propagates user-initiated retractions (`DELETE /curations/{id}`); the per-MERGE guardrail refuses to fire that clause on a suspicious empty/short source.

## Create target tables (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.works.work_author_claim_curations (
    curation_id        STRING NOT NULL,
    user_id            STRING NOT NULL,
    author_id          BIGINT NOT NULL,
    work_id            BIGINT NOT NULL,
    raw_author_name    STRING NOT NULL,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (work_id)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.works.work_author_remove_curations (
    curation_id        STRING NOT NULL,
    user_id            STRING NOT NULL,
    author_id          BIGINT NOT NULL,
    work_id            BIGINT NOT NULL,
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Sync works curations

In [ ]:
%sql
-- Empty-when-target-nonempty fails unconditionally; decline check is overridable.
-- (Both 0 on startup is legitimate and doesn't fail.)
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (
  SELECT COUNT(*) FROM (
    SELECT curation_id FROM openalex_users.public.work_author_claim_curations
    UNION ALL
    SELECT curation_id FROM openalex_users.public.work_author_remove_curations
  )
);
SET VARIABLE current_count = (
  SELECT (SELECT COUNT(*) FROM openalex.works.work_author_claim_curations)
       + (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations)
);

SELECT
  new_count AS new_curations,
  current_count AS current_curations,
  ROUND(new_count * 100.0 / NULLIF(current_count, 0), 2) AS pct_of_current,
  COALESCE(:guardrails_override, 'false') AS guardrails_override;

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 work-author curations but targets hold ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source views before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND LOWER(COALESCE(:guardrails_override, 'false')) <> 'true'
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source curation count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job to bypass.'))
END;

In [ ]:
%sql
SELECT
    'claim'  AS action_kind,
    curation_id, user_id, author_id, work_id,
    raw_author_name,
    created
FROM openalex_users.public.work_author_claim_curations
UNION ALL
SELECT
    'remove' AS action_kind,
    curation_id, user_id, author_id, work_id,
    CAST(NULL AS STRING) AS raw_author_name,
    created
FROM openalex_users.public.work_author_remove_curations

In [ ]:
%sql
MERGE INTO openalex.works.work_author_claim_curations AS target
USING (
    SELECT
        curation_id, user_id, author_id, work_id, raw_author_name, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_author_claim_curations
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [ ]:
%sql
MERGE INTO openalex.works.work_author_remove_curations AS target
USING (
    SELECT
        curation_id, user_id, author_id, work_id, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.work_author_remove_curations
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
SELECT
  (SELECT COUNT(*)              FROM openalex.works.work_author_claim_curations)   AS claim_curation_rows,
  (SELECT COUNT(*)              FROM openalex.works.work_author_remove_curations) AS removal_curation_rows,
  (SELECT MAX(updated_datetime) FROM openalex.works.work_author_claim_curations)   AS claim_last_sync,
  (SELECT MAX(updated_datetime) FROM openalex.works.work_author_remove_curations) AS removal_last_sync

In [ ]:
%sql
SELECT 'claim' AS action_kind,
       curation_id, user_id, author_id, work_id, raw_author_name, created, updated_datetime
FROM openalex.works.work_author_claim_curations
UNION ALL
SELECT 'remove' AS action_kind,
       curation_id, user_id, author_id, work_id,
       CAST(NULL AS STRING) AS raw_author_name,
       created, updated_datetime
FROM openalex.works.work_author_remove_curations
ORDER BY updated_datetime DESC
LIMIT 100